# Making SIMSOPT GPU native: AL conditioning study

Select **Runtime > Change runtime type > GPU**, then run all cells. Four CPU/GPU candidates first run on the engineering problem to isolate inner budget, penalty growth/capping, and per-family constraint scaling. A feasibility-first ranking selects one configuration for a production `stress` confirmation. Every candidate retains complete CPU/GPU final field and coil metrics; the final run also exports VTS/VTU. This workflow is intentionally substantial and may take about an hour on a Colab GPU.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest", "pyevtk"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], cwd=repo, check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-al-conditioning")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/sweep_augmented_lagrangian_conditioning.py", "--screen-problem", "engineering", "--final-problem", "stress", "--maxcor", "100", "--maxls", "50", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output-dir", str(artifact_root)], cwd=repo, env=env, check=True)

In [ ]:
import json

summary = json.loads((artifact_root / "conditioning-study-summary.json").read_text())
assert summary["schema_version"] == 1
assert summary["workflow"] == "augmented_lagrangian_conditioning_study"
assert len(summary["candidates"]) == 4
for candidate in summary["candidates"]:
    result = json.loads((artifact_root / candidate["result_file"]).read_text())
    assert result["schema_version"] == 6
    for backend in ("cpu", "gpu"):
        metrics = result[backend]["final_metrics"]
        assert "objective" in metrics
        assert "normalized_normal_field" in metrics
        assert "coil_constraints" in metrics
        optimization = result[backend]["optimization"]
        assert "final_constraints" in optimization
        assert "final_scaled_constraints" in optimization
final_result = json.loads((artifact_root / summary["final_result_file"]).read_text())
assert final_result["schema_version"] == 6
assert final_result["nvidia_smi"]
stem = Path(summary["final_result_file"]).stem
expected_files = [f"{stem}-cpu_final_surface.vts", f"{stem}-cpu_final_coils.vtu", f"{stem}-gpu_final_surface.vts", f"{stem}-gpu_final_coils.vtu"]
for filename in expected_files:
    path = artifact_root / filename
    assert path.is_file() and path.stat().st_size > 0, path
decision = {"ranking": summary["ranking"], "winner": summary["winner"], "accepted": summary["accepted"], "final_gates": summary["final_acceptance_gates"], "final_cpu_metrics": final_result["cpu"]["final_metrics"], "final_gpu_metrics": final_result["gpu"]["final_metrics"]}
print(json.dumps(decision, indent=2))

## Interpretation

The screening rank prioritizes direct physical feasibility, then worst and total normalized violation, stationarity, field quality, and evaluations. It does not reward a fast but physically worse candidate. Download the archive even if the production confirmation fails; the candidate histories are needed to distinguish insufficient inner convergence from a lossy aggregate-constraint representation.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/simsopt-al-conditioning", "zip", artifact_root)
files.download(archive)